In [0]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Criar uma tabela de Clientes para equipe de marketing:                                  #   #   Importante somente pessoas que tenham e-mail e tefefone cadastrados                     #  
#    Origem camada bronze: customers                                                        #
#    Destino camada silver: customers                                                       #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela bronze                              #
# 2. Criar uma tabela Silver com os dados da tabela bronze                                  #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [0]:
import os
import sys

# Procura automaticamente em qual pasta acima está a 'funcoes_diversas'
caminho = os.getcwd()
while caminho != os.path.dirname(caminho):
    if os.path.exists(os.path.join(caminho, "funcoes_diversas")):
        if caminho not in sys.path:
            sys.path.insert(0, caminho)
        break
    caminho = os.path.dirname(caminho)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria



In [0]:
mapa_criado = gerar_tabela_temporaria(["customers"])

In [0]:
df_customers_silver = spark.sql("""
    select C.customer_id,
           C.first_name,
           C.last_name,
           C.phone,
           C.email,
           C.street,
           C.city,
           C.state,
           C.zip_code           
    from tmp_bronze_customers as C
    where 1 = 1
      and C.email is not null -- vazio
      and C.email not in('NULL', 'NULL ') -- com palavra null
      and C.phone is not null -- vazio
      and C.phone not in('NULL', 'NULL ') -- com palavra null
""")


In [0]:
silver_path = buscar_caminho("silver")
# transformando consulta em uma tabela Silver 
df_customers_silver.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{silver_path}/customers")